# LSTM Autoencoder — SVO Beta Variation Study

Trains the LSTM sequence-to-sequence autoencoder on SVO DMFT data across
varying inverse temperatures **β** (fixed `nwarmup = 150 000`).

**Task:** G₀(iω) + β → Im[Σ(iω)]  
**Data:** `processed_data/data_beta_variation_svo.npz`  
**Architecture:** `model.py` → `build_lstm_autoencoder`  
**Outputs:** `models/`, `results/`, `plots/`

Train/test split strategy: hold out a subset of β values entirely so the
model is evaluated on unseen temperatures (interpolation test).

In [ ]:
import os
import sys
import random
import importlib

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

sys.path.insert(0, os.path.dirname(os.path.abspath("model.py")))
# Force a fresh load so edits to model.py take effect without a kernel restart.
if "model" in sys.modules:
    importlib.reload(sys.modules["model"])
from model import build_lstm_autoencoder

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
DATA_PATH   = "./processed_data/data_beta_variation_svo.npz"
MODELS_DIR  = "./models"
RESULTS_DIR = "./results"
PLOTS_DIR   = "./plots"

BETAS_TEST  = [30, 70]

LATENT_DIM   = 16
DROPOUT_RATE = 0.3
LABEL_DIM    = 2      # [beta_norm, 1/beta_norm]
BATCH_SIZE   = 64
USE_FILM     = True   # FiLM conditioning; set False to revert to concatenation

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR,   exist_ok=True)

## 1. Load and inspect data

In [ ]:
data = np.load(DATA_PATH)

beta_all      = data["beta"]           # shape (N,)  — float, e.g. 20.0 … 100.0
img_g0_all    = data["imgpart_g0"]     # shape (N, n_freq)
img_sigma_all = data["imgpart_sigma"]  # shape (N, n_freq)

sequence_length = img_g0_all.shape[1]
unique_betas    = np.sort(np.unique(beta_all)).astype(int)

print(f"Total samples    : {len(beta_all)}")
print(f"Frequency points : {sequence_length}")
print(f"Beta values      : {unique_betas}")
print(f"Samples per beta : {len(beta_all) // len(unique_betas)}")

## 2. Train / test split by β

Holding out `BETAS_TEST` entirely tests whether the model can interpolate
to unseen temperatures — a harder and more physically meaningful evaluation
than a random sample split.

In [ ]:
beta_int = beta_all.astype(int)

mask_test  = np.isin(beta_int, BETAS_TEST)
mask_train = ~mask_test

betas_train = np.sort(np.unique(beta_int[mask_train]))
betas_test  = np.sort(np.unique(beta_int[mask_test]))

print(f"Train β values : {betas_train}  ({mask_train.sum()} samples)")
print(f"Test  β values : {betas_test}   ({mask_test.sum()} samples)")

## 3. Normalise labels and reshape inputs

In [ ]:
beta_min, beta_max = unique_betas.min(), unique_betas.max()
beta_norm     = (beta_all - beta_min) / (beta_max - beta_min)
inv_beta_norm = 1.0 / (beta_all / beta_min)
label_all    = np.stack([beta_norm, inv_beta_norm], axis=-1).astype("float32")

beta_int_all = beta_all.astype(int)

# ── Train / val / test split by β ─────────────────────────────────────────────
# All 6 spin channels per β are retained. CT-HYB is a Monte Carlo method so
# each channel carries independent stochastic noise — a different random walk
# on the same physical state — providing free data augmentation at the solver
# noise level. Splits are by β value so no temperature leaks across partitions.
BETA_VAL = 50

mask_test_d  = np.isin(beta_int_all, BETAS_TEST)
mask_val_d   = (beta_int_all == BETA_VAL)
mask_train_d = ~mask_test_d & ~mask_val_d

betas_train_d = beta_int_all[mask_train_d]
betas_val_d   = beta_int_all[mask_val_d]
betas_test_d  = beta_int_all[mask_test_d]

print(f"Train β : {sorted(np.unique(betas_train_d))}  ({mask_train_d.sum()} samples)")
print(f"Val   β : {sorted(np.unique(betas_val_d))}    ({mask_val_d.sum()} samples)")
print(f"Test  β : {sorted(np.unique(betas_test_d))}   ({mask_test_d.sum()} samples)")

# ── Normalise using training-set statistics ───────────────────────────────────
sigma_mean = img_sigma_all[mask_train_d].mean()
sigma_std  = img_sigma_all[mask_train_d].std()
g0_mean    = img_g0_all[mask_train_d].mean()
g0_std     = img_g0_all[mask_train_d].std()

def norm_sigma(x):   return (x - sigma_mean) / sigma_std
def denorm_sigma(x): return x * sigma_std + sigma_mean
def norm_g0(x):      return (x - g0_mean) / g0_std

X_train = norm_g0(img_g0_all[mask_train_d]).reshape(-1, sequence_length, 1).astype("float32")
X_val   = norm_g0(img_g0_all[mask_val_d  ]).reshape(-1, sequence_length, 1).astype("float32")
X_test  = norm_g0(img_g0_all[mask_test_d ]).reshape(-1, sequence_length, 1).astype("float32")

Y_train = norm_sigma(img_sigma_all[mask_train_d]).reshape(-1, sequence_length, 1).astype("float32")
Y_val   = norm_sigma(img_sigma_all[mask_val_d  ]).reshape(-1, sequence_length, 1).astype("float32")
Y_test  = norm_sigma(img_sigma_all[mask_test_d ]).reshape(-1, sequence_length, 1).astype("float32")

L_train = label_all[mask_train_d]
L_val   = label_all[mask_val_d  ]
L_test  = label_all[mask_test_d ]

print(f"\nX_train : {X_train.shape}   L_train : {L_train.shape}")
print(f"X_val   : {X_val.shape}     L_val   : {L_val.shape}")
print(f"X_test  : {X_test.shape}    L_test  : {L_test.shape}")
print(f"Σ norm  : mean={sigma_mean:.4f}  std={sigma_std:.4f}")

## 4. Build model

In [ ]:
model = build_lstm_autoencoder(
    sequence_length=sequence_length,
    latent_dim=LATENT_DIM,
    label_dim=LABEL_DIM,
    dropout_rate=DROPOUT_RATE,
    use_film=USE_FILM,
)
model.summary()


## 5. Train

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=40,
    restore_best_weights=True,
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=20,
    min_lr=1e-5,
    verbose=1,
)

history = model.fit(
    [X_train, L_train],
    Y_train,
    epochs=1000,
    batch_size=BATCH_SIZE,
    validation_data=([X_val, L_val], Y_val),
    callbacks=[early_stop, reduce_lr],
    verbose=1,
)
print(f"Stopped at epoch {len(history.history['loss'])}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history.history["loss"],     "b-", label="train loss")
ax.plot(history.history["val_loss"], "r-", label="val loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE loss")
ax.set_yscale("log")
ax.legend()
ax.set_title("Training curve")
fig.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR, "training_loss.png"), dpi=150)
plt.show()

In [ ]:
model_path = os.path.join(MODELS_DIR, "lstm_beta_variation_svo.keras")
model.save(model_path)
print(f"Model saved → {model_path}")

## 6. Evaluate on held-out β values

Average over spin channel samples for each β to get a mean predicted
profile and compare against the mean DMFT reference.

In [ ]:
Y_pred_norm = model.predict([X_test, L_test], verbose=0)
print(f"Predictions shape: {Y_pred_norm.shape}")


In [ ]:
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica'],
    'font.size': 10,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
    'legend.fontsize': 10,
    'axes.linewidth': 2.0,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.minor.width': 0.6,
    'ytick.minor.width': 0.6,
    'xtick.major.size': 5,
    'ytick.major.size': 5,
    'xtick.minor.size': 3,
    'ytick.minor.size': 3,
    'axes.labelpad': 10,
})

freq_full = np.linspace(-161, 161, 2050)
freq_mask = (freq_full > 0) & (freq_full < 10)
freq_sel  = freq_full[freq_mask]

rms = lambda x: np.sqrt(np.mean(x ** 2))

betas_test_arr    = beta_int_all[mask_test]
unique_test_betas = sorted(np.unique(betas_test_arr))

ML_COLORS = [('blue', 'k-'),
             ('red',  'k--')]

rms_per_beta  = []
y_pred_means  = []
y_truth_means = []

os.makedirs(PLOTS_DIR, exist_ok=True)
plt.figure()

for b, (ml_color, dmft_style) in zip(unique_test_betas, ML_COLORS):
    sel = (betas_test_arr == b)

    pred_mean  = denorm_sigma(Y_pred_norm[sel].mean(axis=0).flatten())
    pred_std   = Y_pred_norm[sel].std(axis=0).flatten() * sigma_std
    truth_mean = denorm_sigma(Y_test[sel].mean(axis=0).flatten())

    rel_rms = rms(pred_mean - truth_mean) / (rms(truth_mean) + 1e-12)
    rms_per_beta.append(rel_rms)
    y_pred_means.append(pred_mean)
    y_truth_means.append(truth_mean)

    plt.plot(freq_sel, pred_mean,
                 marker='s', color=ml_color, markersize=6.5,
                 label=f'ML  β={b}')
    plt.plot(freq_sel, truth_mean, dmft_style, zorder=7,linewidth=3,
             label=f'DMFT  β={b}')

plt.ylabel(r'Mean Im $\Sigma_{i\omega}$ (1/eV)', fontsize=20)
plt.xlabel(r'$\omega$ (eV)', fontsize=20)
plt.legend(fontsize=15)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'beta_predictions_svo.eps'),
            format='eps', dpi=600, bbox_inches='tight')
plt.show()

for b, r in zip(unique_test_betas, rms_per_beta):
    print(f'  β = {b:3d}  rel. RMS = {r:.5f}')


## 7. Check performance on training β values

One representative sample per training β to verify the model hasn't
overfit or collapsed.

In [ ]:
Y_pred_train_norm = model.predict([X_train, L_train], verbose=0)

unique_train_betas = sorted(np.unique(betas_train_d))
rms_train_per_beta = []

for b in unique_train_betas:
    sel = (betas_train_d == b)
    pred_mean  = denorm_sigma(Y_pred_train_norm[sel].mean(axis=0).flatten())
    pred_std   = Y_pred_train_norm[sel].std(axis=0).flatten() * sigma_std
    truth_mean = denorm_sigma(Y_train[sel].mean(axis=0).flatten())
    rel_rms    = rms(pred_mean - truth_mean) / (rms(truth_mean) + 1e-12)
    rms_train_per_beta.append(rel_rms)

    plt.figure(figsize=(8, 5))
    plt.errorbar(freq_sel, pred_mean, yerr=pred_std,
                 marker='s', color='blue', markersize=5, label='ML')
    plt.plot(freq_sel, truth_mean, 'k-', zorder=3, label='DMFT')
    plt.title(f'SVO  β = {b}', fontsize=16)
    plt.ylabel(r'Mean Im $\Sigma_{i\omega}$ (1/eV)', fontsize=20)
    plt.xlabel(r'$\omega$ (eV)', fontsize=20)
    plt.legend(fontsize=15)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, f'train_beta_{b}_prediction.png'),
                format='png', dpi=600, bbox_inches='tight')
    plt.show()
    print(f'  β = {b:3d}  rel. RMS = {rel_rms:.5f}')


## 8. RMS residual vs β

In [ ]:
betas_test_sorted  = unique_test_betas
betas_train_sorted = unique_train_betas

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(betas_train_sorted, rms_train_per_beta, "bs-", label="train β",    markersize=7)
ax.plot(betas_test_sorted,  rms_per_beta,       "r*",  label="held-out β", markersize=12)
ax.set_xlabel(r"$\beta$ (1/eV)", fontsize=13)
ax.set_ylabel("Relative RMS residual", fontsize=13)
ax.set_title(r"Model accuracy vs $\beta$ (SVO)")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR, "rms_vs_beta.png"), dpi=150)
plt.show()

print("\nSummary — relative RMS residual per β:")
print(f"  Training β  : { {b: f'{r:.5f}' for b, r in zip(betas_train_sorted, rms_train_per_beta)} }")
print(f"  Held-out β  : { {b: f'{r:.5f}' for b, r in zip(betas_test_sorted,  rms_per_beta)} }")

## 9. Baseline: linear interpolation in β-space

The dumbest possible interpolator: for each held-out β, take the weighted
average of the two nearest training neighbours. If the LSTM doesn't beat
this, the architecture is providing no value over a lookup table.

In [ ]:
print(f"{'β':>6}  {'LSTM rel.RMS':>14}  {'LinInterp rel.RMS':>18}  {'LSTM wins?':>10}")
print("-" * 56)

train_arr    = np.array(betas_train_sorted)
rms_baseline = []

for b, lstm_rms in zip(sorted(np.unique(betas_test_d)), rms_per_beta):
    lower = train_arr[train_arr < b]
    upper = train_arr[train_arr > b]
    b_lo  = lower[-1] if len(lower) else upper[0]
    b_hi  = upper[0]  if len(upper) else lower[-1]

    s_lo = img_sigma_all[beta_int_all == b_lo].mean(axis=0)
    s_hi = img_sigma_all[beta_int_all == b_hi].mean(axis=0)
    w    = (b - b_lo) / (b_hi - b_lo) if b_lo != b_hi else 0.0
    interp = (1 - w) * s_lo + w * s_hi

    truth    = img_sigma_all[beta_int_all == b].mean(axis=0)
    base_rms = rms(interp - truth) / (rms(truth) + 1e-12)
    rms_baseline.append(base_rms)

    wins = "YES" if lstm_rms < base_rms else "no"
    print(f"  β={b:3d}  {lstm_rms:14.5f}  {base_rms:18.5f}  {wins:>10}")